In [1]:
!pip install yfinance


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, mean_absolute_error

In [3]:
# Last 6 months of data
df = yf.download("AAPL", period="6mo", interval="1d")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df.reset_index()
df = df.sort_values("Date")

[*********************100%***********************]  1 of 1 completed


In [4]:
# --- Feature Engineering ---

df["sma_10"] = df["Close"].rolling(10).mean()
df["sma_20"] = df["Close"].rolling(20).mean()

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

rolling_mean = df["Close"].rolling(20).mean()
rolling_std  = df["Close"].rolling(20).std()
df["bb_upper"] = rolling_mean + (2 * rolling_std)
df["bb_lower"] = rolling_mean - (2 * rolling_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

df["return"]      = df["Close"].pct_change()
df["return_3d"]   = df["Close"].pct_change(3)
df["return_5d"]   = df["Close"].pct_change(5)

df["volatility_5d"]  = df["return"].rolling(5).std()
df["volatility_10d"] = df["return"].rolling(10).std()

df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]

# Targets
df["target_signal"] = (df["return"].shift(-1) > 0.002).astype(int)
df["target_price"]  = df["Close"].shift(-1)

df = df.dropna()

In [5]:
feature_cols = [
    "rsi_14", "macd", "macd_signal", "bb_width",
    "return_3d", "return_5d", "volatility_5d", "volatility_10d",
    "dist_sma_10", "dist_sma_20", "Volume"
]

X        = df[feature_cols].values
y_signal = df["target_signal"].values
y_price  = df["target_price"].values

split_index = int(len(df) * 0.8)

In [6]:
# Scale features and price target
feat_scaler  = StandardScaler()
price_scaler = StandardScaler()

X_scaled       = feat_scaler.fit_transform(X)
y_price_scaled = price_scaler.fit_transform(y_price.reshape(-1, 1)).flatten()

# Build sequences with lookback = 10
LOOKBACK = 10

def create_sequences(X, y_sig, y_prc, lookback):
    Xs, ys_sig, ys_prc = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback:i])
        ys_sig.append(y_sig[i])
        ys_prc.append(y_prc[i])
    return np.array(Xs), np.array(ys_sig), np.array(ys_prc)

X_seq, y_sig_seq, y_prc_seq = create_sequences(X_scaled, y_signal, y_price_scaled, LOOKBACK)

train_size = split_index - LOOKBACK

X_train = torch.tensor(X_seq[:train_size], dtype=torch.float32)
X_test  = torch.tensor(X_seq[train_size:], dtype=torch.float32)

y_sig_train = torch.tensor(y_sig_seq[:train_size], dtype=torch.float32)
y_sig_test  = torch.tensor(y_sig_seq[train_size:], dtype=torch.float32)

y_prc_train = torch.tensor(y_prc_seq[:train_size], dtype=torch.float32)
y_prc_test  = torch.tensor(y_prc_seq[train_size:], dtype=torch.float32)

train_ds = TensorDataset(X_train, y_sig_train, y_prc_train)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=False)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: torch.Size([74, 10, 11]), Test: torch.Size([21, 10, 11])


In [7]:
# --- Dual-output LSTM model ---
class DualLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.drop1 = nn.Dropout(0.3)
        self.lstm2 = nn.LSTM(hidden_size, 32, batch_first=True)
        self.drop2 = nn.Dropout(0.3)
        self.fc    = nn.Linear(32, 16)
        self.relu  = nn.ReLU()

        # Head 1: signal (BUY/SELL/HOLD)
        self.signal_head = nn.Sequential(nn.Linear(16, 1), nn.Sigmoid())
        # Head 2: next day closing price
        self.price_head  = nn.Linear(16, 1)

    def forward(self, x):
        x, _ = self.lstm1(x)
        x = self.drop1(x)
        x, _ = self.lstm2(x)
        x = self.drop2(x[:, -1, :])   # take last timestep
        x = self.relu(self.fc(x))
        return self.signal_head(x).squeeze(-1), self.price_head(x).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = DualLSTM(input_size=len(feature_cols)).to(device)
print(model)

c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10\cuda\CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


DualLSTM(
  (lstm1): LSTM(11, 64, batch_first=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (lstm2): LSTM(64, 32, batch_first=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=32, out_features=16, bias=True)
  (relu): ReLU()
  (signal_head): Sequential(
    (0): Linear(in_features=16, out_features=1, bias=True)
    (1): Sigmoid()
  )
  (price_head): Linear(in_features=16, out_features=1, bias=True)
)


In [8]:
optimizer   = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_bce    = nn.BCELoss()
loss_mse    = nn.MSELoss()

best_val_loss = float("inf")
patience, wait = 10, 0
best_weights   = None

EPOCHS = 100
X_test_dev     = X_test.to(device)
y_sig_test_dev = y_sig_test.to(device)
y_prc_test_dev = y_prc_test.to(device)

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, ysb, ypb in train_dl:
        xb, ysb, ypb = xb.to(device), ysb.to(device), ypb.to(device)
        sig_out, prc_out = model(xb)
        loss = loss_bce(sig_out, ysb) + loss_mse(prc_out, ypb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Validation loss
    model.eval()
    with torch.no_grad():
        sig_val, prc_val = model(X_test_dev)
        val_loss = (loss_bce(sig_val, y_sig_test_dev) + loss_mse(prc_val, y_prc_test_dev)).item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | val_loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights  = {k: v.clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_weights)

Epoch  10 | val_loss: 1.6542
Early stopping at epoch 11


<All keys matched successfully>

In [9]:
model.eval()
with torch.no_grad():
    sig_prob_t, price_pred_t = model(X_test_dev)

sig_prob    = sig_prob_t.cpu().numpy()
sig_pred    = (sig_prob > 0.5).astype(int)
price_pred  = price_scaler.inverse_transform(price_pred_t.cpu().numpy().reshape(-1, 1)).flatten()
price_actual = price_scaler.inverse_transform(y_prc_test.numpy().reshape(-1, 1)).flatten()

print("LSTM Signal Accuracy:", accuracy_score(y_sig_test.numpy(), sig_pred))
print(classification_report(y_sig_test.numpy(), sig_pred))
print("LSTM ROC-AUC:", roc_auc_score(y_sig_test.numpy(), sig_prob))
print("LSTM Price MAE: $", round(mean_absolute_error(price_actual, price_pred), 2))

LSTM Signal Accuracy: 0.6190476190476191
              precision    recall  f1-score   support

         0.0       0.62      1.00      0.76        13
         1.0       0.00      0.00      0.00         8

    accuracy                           0.62        21
   macro avg       0.31      0.50      0.38        21
weighted avg       0.38      0.62      0.47        21

LSTM ROC-AUC: 0.27884615384615385
LSTM Price MAE: $ 5.72


c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [10]:
# --- Combined output table ---
results = pd.DataFrame({
    "actual_next_close":    price_actual,
    "predicted_next_close": price_pred,
    "actual_signal":        y_sig_test.numpy(),
    "predicted_signal":     sig_pred
})
print(results.tail(10))

    actual_next_close  predicted_next_close  actual_signal  predicted_signal
11         264.720001            266.493805            1.0                 0
12         263.750000            266.507507            0.0                 0
13         262.519989            266.516144            0.0                 0
14         260.290009            266.513214            0.0                 0
15         257.459991            266.497925            0.0                 0
16         259.880005            266.479034            1.0                 0
17         260.829987            266.459106            1.0                 0
18         260.809998            266.458282            0.0                 0
19         255.759995            266.471527            0.0                 0
20         250.119995            266.493164            0.0                 0


In [11]:
# --- Latest signal + next day price forecast ---
prob            = float(sig_prob[-1])
predicted_price = float(price_pred[-1])

if prob > 0.6:
    signal = "BUY"
elif prob < 0.4:
    signal = "SELL"
else:
    signal = "HOLD"

print(f"Signal:               {signal}")
print(f"Confidence:           {round(prob, 2)}")
print(f"Predicted Next Close: ${round(predicted_price, 2)}")

Signal:               HOLD
Confidence:           0.49
Predicted Next Close: $266.49
